In [ ]:
import myModule as mine
import time
import shutil
import numpy as np
import matplotlib.pyplot as plt
# import Unified_cl_methods as ucm
import MODE_utilities as AS
import Case1 as c1
import pygad
import os
from scipy.stats import zscore

In [ ]:
TYPE_OF_DATA = 'cpu'
SPLIT = 0.80
split_data = AS.read_raspberry_pi_dataset(
    series_length=500, column=TYPE_OF_DATA
)

split_data = split_data[:60][:]

split_data


offline_data = split_data[:, :int(SPLIT*split_data.shape[1])]
online_data = split_data[:, int(SPLIT*split_data.shape[1]):]

In [ ]:
X_scaled = c1.minmax_scale(offline_data)
X_scaled

In [ ]:
offline_scaled_conc = np.concatenate([ts for ts in X_scaled])
offline_scaled_conc

In [ ]:
gene_space = [
    {'low': 2, 'high': 10, 'step': 1},   # hidden_dim
    {'low': 1, 'high': 5, 'step': 1},     # layer_dim
    {'low': 5, 'high': 20, 'step': 1},    # window
    {'low': 8, 'high': 128, 'step': 1},      # batch_size
    {'low': 50, 'high': 150, 'step': 1},      # epochs
    {'low': 0.0001, 'high': 0.1},           # learning_rate
]


In [ ]:

class Simple_GA:
    def __init__(self, gene_space, offline_scaled_conc, online_data):
        
        self.DATA = offline_scaled_conc
        self.gene_space = gene_space
        self.online_ts = online_data
        
        

        start = time.time()
        self.best_model = None
        self.best_solution = None
        self.best_fitness = float("inf")
        self.best_instance = None
        self.result = []
        self.training_duration = None
        self.run()
        self.duration = time.time() - start
        self.evaluate_online(self.online_ts)




    def fitness_func(self, ga_instance, genes, genes_idx):
        chromosome = {
            'hidden_dim': int(genes[0]),
            'layer_dim': int(genes[1]),
            'window': int(genes[2]),
            'batch_size': int(genes[3]),
            'epochs': int(genes[4]),
            'learning_rate': genes[5]
        }
        
        instance = c1.Instance(self.DATA, chromosome)
        instance.train_phase()
        rmse, _, _, _ = instance.validation_phase()

        if rmse < self.best_fitness:
            self.best_fitness = rmse
            self.best_solution = genes.copy()
            self.best_model = instance.model
            self.best_instance = instance
        return -rmse
    

    def run(self, generations=11, population_size=10):
        ga_instance = pygad.GA(
            num_generations=generations,
            num_parents_mating=10,
            fitness_func=self.fitness_func,
            sol_per_pop=population_size,
            num_genes=len(self.gene_space),
            gene_space=self.gene_space,
            mutation_percent_genes=50,
            mutation_type="random",
            crossover_type="single_point",
            allow_duplicate_genes=True,
            keep_elitism=5
        )
       
        ga_instance.run()
       

    def evaluate_online(self, onlinedata):
        with open(f'GA_Rasbery_Pi_cpu.txt', 'w') as f:
            f.write(f"General info\n=============\n")
            f.write(f"Total optimization time: {self.duration}, Training size: {len(self.DATA)}\n")
        for idx, ts in enumerate(onlinedata):
            rmse, _, _, _, _ = c1.evaluate_on_new_data(
                trained_model=self.best_instance.model,
                new_data=ts,
                chromosome=self.best_instance.chromosome)
            with open(f'GA_Rasbery_Pi_cpu.txt', 'a') as f:
                f.write(f"Devices: {idx} | RMSE: {rmse.item()}\n")

    


In [ ]:
Simple_GA(gene_space, offline_scaled_conc, online_data)